
# Wikipedia Passage Retrieval with SentenceTransformer + FAISS


In [ ]:

from pathlib import Path
from urllib.parse import urlparse, unquote
import json
import re
import time
import random

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import faiss

DATA_PATH = Path("../Data/wiki_sources.csv")

OUTPUT_DIR = Path("../WIKI_resource/wiki_retrieval_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CACHE_PATH = OUTPUT_DIR / "wiki_pages_cache.jsonl"
PAGES_CSV_PATH = OUTPUT_DIR / "wiki_pages.csv"
PASSAGES_CSV_PATH = OUTPUT_DIR / "wiki_passages.csv"
FAISS_PATH = OUTPUT_DIR / "wiki_passages.faiss"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TARGET_WORDS = 150
OVERLAP_WORDS = 30
TOP_K = 5

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())



## 1. Load and canonicalize Wikipedia URLs


In [ ]:

sources_df = pd.read_csv(DATA_PATH)

if "Wikipedia_URL" not in sources_df.columns:
    raise ValueError("wiki_sources.csv must contain a Wikipedia_URL column.")

print("Rows:", len(sources_df))
display(sources_df.head())


In [ ]:

def parse_wikipedia_url(url):
    if pd.isna(url):
        return None

    url = str(url).strip()
    parsed = urlparse(url)
    host = parsed.netloc.lower()

    if not host.endswith(".wikipedia.org"):
        return None
    if not parsed.path.startswith("/wiki/"):
        return None

    raw_title = parsed.path[len("/wiki/"):]
    if not raw_title:
        return None

    title = unquote(raw_title).replace("_", " ")
    section = unquote(parsed.fragment).replace("_", " ") if parsed.fragment else ""
    canonical_url = f"https://{host}/wiki/{raw_title}"

    return {
        "host": host,
        "page_title_from_url": title,
        "section_from_url": section,
        "canonical_url": canonical_url,
    }


parsed_rows = []

for _, row in sources_df.iterrows():
    info = parse_wikipedia_url(row["Wikipedia_URL"])
    if info is None:
        continue

    parsed_rows.append({
        "Question_ID": row.get("Question_ID", None),
        "original_url": row["Wikipedia_URL"],
        **info,
    })

parsed_df = pd.DataFrame(parsed_rows)

print("Valid Wikipedia URL rows:", len(parsed_df))
print("Unique canonical pages:", parsed_df["canonical_url"].nunique())
display(parsed_df.head())


In [ ]:

unique_pages_df = (
    parsed_df[["host", "page_title_from_url", "canonical_url"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_pages_df["page_key"] = (
    unique_pages_df["host"] + "::" + unique_pages_df["page_title_from_url"]
)

print("Unique pages to download:", len(unique_pages_df))
display(unique_pages_df.head(10))



## 2. Download Wikipedia page text


In [ ]:

HEADERS = {
    "User-Agent": "AcademicHallucinationRetrievalProject/0.1 (educational research)"
}

session = requests.Session()
session.headers.update(HEADERS)


def api_url_for_host(host):
    return f"https://{host}/w/api.php"


def load_cache(path):
    cache = {}
    if not path.exists():
        return cache

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            cache[obj["page_key"]] = obj

    return cache


def append_cache(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


page_cache = load_cache(RAW_CACHE_PATH)
print("Already cached pages:", len(page_cache))


In [ ]:

def fetch_wikipedia_page(host, title, max_retries=6):
    endpoint = api_url_for_host(host)

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "redirects": 1,
        "titles": title,
        "format": "json",
        "formatversion": 2,
    }

    for attempt in range(max_retries):
        try:
            response = session.get(endpoint, params=params, timeout=30)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait = float(retry_after) if retry_after else min(60, (2 ** attempt) + random.random())
                print(f"429 for {title!r}. Sleep {wait:.1f}s...")
                time.sleep(wait)
                continue

            if response.status_code >= 500:
                wait = min(60, (2 ** attempt) + random.random())
                print(f"Server error {response.status_code} for {title!r}. Retry in {wait:.1f}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()
            data = response.json()

            pages = data.get("query", {}).get("pages", [])
            if not pages:
                return {
                    "status": "missing",
                    "resolved_title": title,
                    "page_id": None,
                    "text": "",
                }

            page = pages[0]

            if page.get("missing") is True:
                return {
                    "status": "missing",
                    "resolved_title": page.get("title", title),
                    "page_id": page.get("pageid"),
                    "text": "",
                }

            return {
                "status": "ok",
                "resolved_title": page.get("title", title),
                "page_id": page.get("pageid"),
                "text": page.get("extract", "") or "",
            }

        except requests.RequestException as e:
            if attempt == max_retries - 1:
                return {
                    "status": "error",
                    "resolved_title": title,
                    "page_id": None,
                    "text": "",
                    "error": str(e),
                }

            wait = min(60, (2 ** attempt) + random.random())
            print(f"Request error for {title!r}. Retry in {wait:.1f}s...")
            time.sleep(wait)

    return {
        "status": "error",
        "resolved_title": title,
        "page_id": None,
        "text": "",
        "error": "max retries exceeded",
    }


In [ ]:

downloaded_records = []

for row in tqdm(
    unique_pages_df.itertuples(index=False),
    total=len(unique_pages_df),
    desc="Downloading Wikipedia pages"
):
    key = row.page_key

    if key in page_cache:
        downloaded_records.append(page_cache[key])
        continue

    result = fetch_wikipedia_page(
        host=row.host,
        title=row.page_title_from_url,
    )

    record = {
        "page_key": key,
        "host": row.host,
        "requested_title": row.page_title_from_url,
        "canonical_url": row.canonical_url,
        **result,
    }

    page_cache[key] = record
    append_cache(RAW_CACHE_PATH, record)
    downloaded_records.append(record)

    time.sleep(0.7)

pages_df = pd.DataFrame(downloaded_records)

print(pages_df["status"].value_counts(dropna=False))
print("Pages with text:", (pages_df["text"].fillna("").str.len() > 0).sum())

pages_df.to_csv(PAGES_CSV_PATH, index=False)
display(pages_df.head())



## 3. Clean text and create passages


In [ ]:

def clean_wikipedia_text(text):
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"(?m)^=+\s*(.*?)\s*=+$", r"\1", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_into_passages(text, target_words=150, overlap_words=30):
    text = clean_wikipedia_text(text)
    if not text:
        return []

    raw_blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
    useful_blocks = [b for b in raw_blocks if len(b.split()) >= 8]

    words = " ".join(useful_blocks).split()
    if not words:
        return []

    if overlap_words >= target_words:
        raise ValueError("overlap_words must be smaller than target_words")

    step = target_words - overlap_words
    passages = []

    for start in range(0, len(words), step):
        chunk = words[start:start + target_words]

        if len(chunk) < 30:
            break

        passages.append(" ".join(chunk))

        if start + target_words >= len(words):
            break

    return passages


In [ ]:

passage_records = []
global_passage_id = 0

ok_pages = pages_df[
    (pages_df["status"] == "ok") &
    (pages_df["text"].fillna("").str.len() > 0)
].copy()

for row in tqdm(
    ok_pages.itertuples(index=False),
    total=len(ok_pages),
    desc="Creating passages"
):
    page_passages = split_into_passages(
        row.text,
        target_words=TARGET_WORDS,
        overlap_words=OVERLAP_WORDS,
    )

    for local_id, passage_text in enumerate(page_passages):
        passage_records.append({
            "passage_id": global_passage_id,
            "page_passage_id": local_id,
            "page_key": row.page_key,
            "page_id": row.page_id,
            "page_title": row.resolved_title,
            "host": row.host,
            "source_url": row.canonical_url,
            "passage_text": passage_text,
            "word_count": len(passage_text.split()),
        })
        global_passage_id += 1

passages_df = pd.DataFrame(passage_records)

print("Pages:", len(ok_pages))
print("Passages:", len(passages_df))
print("Average words/passage:", round(passages_df["word_count"].mean(), 1))

passages_df.to_csv(PASSAGES_CSV_PATH, index=False)
display(passages_df.head())



## 4. Encode passages with SentenceTransformer

In [ ]:

model = SentenceTransformer(MODEL_NAME)

passage_embeddings = model.encode(
    passages_df["passage_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

passage_embeddings = np.asarray(passage_embeddings, dtype="float32")

print("Embedding matrix shape:", passage_embeddings.shape)



## 5. Build the FAISS index


In [ ]:

dimension = passage_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(passage_embeddings)

print("FAISS dimension:", dimension)
print("Vectors in index:", index.ntotal)

faiss.write_index(index, str(FAISS_PATH))
print("Saved:", FAISS_PATH)



## 6. Search evidence for an atomic claim


In [ ]:

def search_claim(claim, k=5, model=model, index=index, passages_df=passages_df):
    query_embedding = model.encode(
        [claim],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    query_embedding = np.asarray(query_embedding, dtype="float32")

    k = min(k, index.ntotal)
    scores, indices = index.search(query_embedding, k)

    results = passages_df.iloc[indices[0]].copy()
    results.insert(0, "score", scores[0])

    return results[
        [
            "score",
            "passage_id",
            "page_title",
            "source_url",
            "passage_text",
        ]
    ].reset_index(drop=True)


In [ ]:
example_claim = "There is no single language that all people in Europe speak."

results = search_claim(example_claim, k=TOP_K)

print("CLAIM:")
print(example_claim)
print("\nTOP EVIDENCE:")
display(results)
